# Day 24: Writing Files and `pathlib`

**Today's question:** how can a program **remember** something after it stops running?

Before running a cell marked 🤔, predict what it will show.

*This notebook only creates **new** files (`greeting.txt`, `no_newlines.txt`, `trap_demo.txt` and an `output`
folder) next to itself. Delete them whenever you like.*

## 1. Why write files?

Variables live in memory. Restart the kernel (or close the program) and they are gone. A file on disk survives.
Yesterday we **read** files; today we **write** them: saving results, keeping a log, building a journal.

## 2. Writing with mode `"w"`

`open()` takes a second argument, the **mode**. Yesterday we used the default, `"r"` (read). `"w"` means write.

In [ ]:
with open("greeting.txt", "w", encoding="utf-8") as out_file:
    out_file.write("Hello from Python!\n")
    out_file.write("This line was written by a program.\n")

print("Saved.")

The file now exists next to this notebook (look in the VS Code explorer). Read it back with yesterday's code:

In [ ]:
with open("greeting.txt", encoding="utf-8") as in_file:
    print(in_file.read())

🤔 **Predict:** what will this file contain? Look closely at what is *missing*.

In [ ]:
with open("no_newlines.txt", "w", encoding="utf-8") as out_file:
    out_file.write("apples")
    out_file.write("bananas")
    out_file.write("cherries")

with open("no_newlines.txt", encoding="utf-8") as in_file:
    print(in_file.read())

`.write()` writes **exactly** the text you give it. Unlike `print()`, it never adds a newline. You add `"\n"` yourself.

Writing a list, one item per line:

In [ ]:
items = ["apples", "bananas", "cherries"]

with open("no_newlines.txt", "w", encoding="utf-8") as out_file:
    for item in items:
        out_file.write(f"{item}\n")

with open("no_newlines.txt", encoding="utf-8") as in_file:
    print(in_file.read())

## 3. `"w"` versus `"a"`: overwrite or append

We create a **new** practice file with three important notes.

In [ ]:
with open("trap_demo.txt", "w", encoding="utf-8") as notes_file:
    notes_file.write("Important note 1\n")
    notes_file.write("Important note 2\n")
    notes_file.write("Important note 3\n")

with open("trap_demo.txt", encoding="utf-8") as notes_file:
    print(notes_file.read())

🤔 **Predict:** we open the **same** file with `"w"` again and write one more note. What does the file contain now?

In [ ]:
with open("trap_demo.txt", "w", encoding="utf-8") as notes_file:
    notes_file.write("Note 4\n")

with open("trap_demo.txt", encoding="utf-8") as notes_file:
    print(notes_file.read())

Notes 1–3 are **gone**. Mode `"w"` empties the file the moment it is opened: no warning, no undo, no recycle bin.

Mode `"a"` (append) adds to the end instead:

In [ ]:
with open("trap_demo.txt", "a", encoding="utf-8") as notes_file:
    notes_file.write("Note 5\n")

with open("trap_demo.txt", encoding="utf-8") as notes_file:
    print(notes_file.read())

| Mode | Meaning | If the file exists | If it does not exist |
|------|---------|--------------------|----------------------|
| `"r"` (default) | read | read from the start | `FileNotFoundError` |
| `"w"` | write | **emptied first** ⚠️ | created |
| `"a"` | append | add to the end | created |

Rule: use `"w"` only when you **mean** "replace everything". For logs, journals and history, use `"a"`.

## 4. `pathlib.Path`: paths as objects

Python comes with a **standard library** of ready-made tools. We used one on Day 14 with `import random`.
Today we take one tool, `Path`, from the `pathlib` module. (`from module import tool` is explained fully on Day 26.)

In [ ]:
from pathlib import Path

output_dir = Path("output")
notes_path = output_dir / "notes.txt"
print(notes_path)

The `/` operator **joins** path parts. It is not division here: Python uses the right separator for your
computer (`/` on Mac and Linux, `\` on Windows), so the same code works everywhere.

🤔 **Predict:** does the `output` folder exist yet?

In [ ]:
print("Folder exists:", output_dir.exists())
print("File exists:", notes_path.exists())

Creating a `Path` does **not** create anything on disk. It is only a description of a location.
`.mkdir()` creates the folder; `exist_ok=True` means "and do not complain if it is already there".

In [ ]:
output_dir.mkdir(exist_ok=True)
print("Folder exists:", output_dir.exists())

`.write_text()` and `.read_text()` do *open → write/read → close* in one line each.

In [ ]:
notes_path.write_text("Buy a new notebook\nEmail Mei about the project\n", encoding="utf-8")

print(notes_path.read_text(encoding="utf-8"))

⚠️ `.write_text()` behaves like mode `"w"`: it **replaces** the whole file.
To append, use `open(path, "a", ...)`, which accepts a `Path` too.

In [ ]:
with open(notes_path, "a", encoding="utf-8") as notes_file:
    notes_file.write("Revise comprehensions\n")

print(notes_path.read_text(encoding="utf-8"))

A `Path` knows its parts:

In [ ]:
print("Name:", notes_path.name)
print("Suffix:", notes_path.suffix)
print("Folder:", notes_path.parent)

## 5. The working directory

Relative paths like `Path("output")` start from the **working directory**. `Path.cwd()` shows it.
The result is different on every computer.

In [ ]:
print(Path.cwd())

In a notebook this is the notebook's folder. When you run `uv run journal.py` in a terminal, it is the folder the
terminal is in, so the same script can create `output/` in a different place.

## 6. Common mistakes (these cells fail **on purpose**)

In [ ]:
# Mistake 1: .write() only accepts text
total = 125.5
with open("trap_demo.txt", "a", encoding="utf-8") as notes_file:
    notes_file.write(total)

In [ ]:
# Mistake 2: writing into a folder that does not exist
report_path = Path("reports") / "summary.txt"
report_path.write_text("All good\n", encoding="utf-8")

In [ ]:
# Mistake 3: creating a folder that already exists, without exist_ok=True
output_dir.mkdir()

In [ ]:
# Mistake 4: forgetting the mode. The default is "r", read only
with open(notes_path, encoding="utf-8") as notes_file:
    notes_file.write("One more note\n")

| Error | Fix |
|-------|-----|
| `TypeError: write() argument must be str, not float` | `f"{total}\n"` |
| `FileNotFoundError` when **writing** | create the folder first: `.mkdir(exist_ok=True)` |
| `FileExistsError` | `.mkdir(exist_ok=True)` |
| `UnsupportedOperation: not writable` | pass `"w"` or `"a"` |

## 7. Real-world example: a journal that remembers

Three small functions. `add_entry` appends, so the journal grows every time we use it.

In [ ]:
def add_entry(path, date, text):
    """Append one entry to the journal file as 'date | text'."""
    path.parent.mkdir(exist_ok=True)
    with open(path, "a", encoding="utf-8") as journal_file:
        journal_file.write(f"{date} | {text}\n")


def read_entries(path):
    """Return the journal entries as a list of lines (an empty list if there is no journal yet)."""
    if not path.exists():
        return []
    lines = path.read_text(encoding="utf-8").split("\n")
    return [line for line in lines if line != ""]


def show_journal(path):
    """Print every journal entry with a number."""
    entries = read_entries(path)
    print(f"Journal: {len(entries)} entries")
    for number, entry in enumerate(entries, start=1):
        print(f"{number}. {entry}")

In [ ]:
journal_path = Path("output") / "journal.txt"
show_journal(journal_path)

In [ ]:
add_entry(journal_path, "2026-11-12", "Learned the difference between w and a")
add_entry(journal_path, "2026-11-12", "The / operator joins folder and file names")
show_journal(journal_path)

Now with the user's own words:

In [ ]:
text = input("New journal entry: ")
add_entry(journal_path, "2026-11-12", text)
show_journal(journal_path)

**Try this:** restart the kernel, run only the `import` cell and the function cell, then `show_journal(journal_path)`.
The entries are still there. The program forgot everything, the **file** did not.

## Recap

1. `open(path, "w")` creates or **empties** a file; `open(path, "a")` adds to the end. `.write()` takes **text only** and adds no newline.
2. `from pathlib import Path`: `Path("output") / "notes.txt"` builds a path; `.exists()` checks it; `.mkdir(exist_ok=True)` creates a folder.
3. `.write_text()` (replaces the file) and `.read_text()` read or write a whole file in one line.
4. Relative paths start from the working directory (`Path.cwd()`).

**Next time:** what to do when things go wrong: bad input, missing files, and `try`/`except`.